In [1]:
import os
import numpy as np
import librosa
import librosa.display
import random

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping

dataset_path = "C:/Users/Yogesh/Desktop/emotional_classification/emotion_project/data/RAVDESS"

In [2]:
# Six emotions: neutral, calm, happy, sad, angry, fearful
def extract_emotion(filename):
    emotion_code = filename.split("-")[2]
    emotion_dict = {
        "01": "neutral",
        "02": "calm",
        "03": "happy",
        "04": "sad",
        "05": "angry",
        "06": "fearful"
    }
    return emotion_dict.get(emotion_code)

def extract_mfcc(file_path, n_mfcc=40, max_len=174):
    audio, sr = librosa.load(file_path, duration=3, offset=0.5)
    
    # Augmentation: randomly apply pitch shift or time stretch
    if random.random() < 0.3:
        audio = librosa.effects.pitch_shift(audio, sr, n_steps=random.uniform(-2,2))
    if random.random() < 0.3:
        rate = random.uniform(0.9, 1.1)
        audio = librosa.effects.time_stretch(audio, rate)
    
    mfcc = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=n_mfcc)
    
    # Normalize MFCC
    mfcc = (mfcc - np.mean(mfcc)) / (np.std(mfcc)+1e-9)
    
    # Pad or truncate to max_len
    if mfcc.shape[1] < max_len:
        pad_width = max_len - mfcc.shape[1]
        mfcc = np.pad(mfcc, ((0,0),(0,pad_width)), mode='constant')
    else:
        mfcc = mfcc[:, :max_len]
    
    return mfcc

In [3]:
X = []
y = []

count = 0

for root, dirs, files in os.walk(dataset_path):
    for file in files:
        if file.endswith(".wav"):
            emotion = extract_emotion(file)
            
            if emotion is not None:   # Only 6 emotions
                file_path = os.path.join(root, file)
                
                mfcc = extract_mfcc(file_path)
                
                X.append(mfcc)
                y.append(emotion)
                
                count += 1
                if count % 200 == 0:
                    print("Processed:", count)

print("Finished Loading")

# Convert to numpy
X = np.array(X)
y = np.array(y)

# Add channel dimension for CNN
X = X[..., np.newaxis]

print("Data Loaded!")
print("X shape:", X.shape)   
print("y shape:", y.shape)

TypeError: time_stretch() takes 1 positional argument but 2 were given